# MandiaCompact — étape11 : source française et répétitions

Expérience à **20,86 M de paramètres** : CE témoin, contraste sur la source, puis contraste + pénalité des boucles. **3 ×300 mises à jour**, mêmes données et passages auxiliaires. Sélection sur validation ; étape0 possible si aucune amélioration. Aucun résultat favorable présumé.

39 nouveaux tests réservés (30 traductions,6 répliques écrites,3 questions pratiques construites). Cibles baoulé publiées dans [Google SMOL](https://huggingface.co/datasets/google/smol), français traduit/adapté par assistant : **pas de validation native ni de conversations enregistrées**.

Tous les checkpoints/états AdamW et RNG restent dans Drive. Ce notebook utilise les dossiers des étapes4–10 déjà présents dans le même Drive. Ne pas relancer toutes les étapes antérieures. La voix sera abordée après l'évaluation du texte.

Sources et limites : METHODE-ET-SOURCES.txt dans le dossier de cette étape. [Unlikelihood](https://arxiv.org/abs/1908.04319) ; [apprentissage contrastif en dialogue](https://aclanthology.org/2020.findings-emnlp.70/). Adaptation expérimentale, pas reproduction exacte ni nouveauté revendiquée.

## Résultats réels — 3 ×300 étapes sur L4,20,86 M de paramètres

**Répetitions réduites sur certains tests, compréhension et dialogues encore insuffisants. Aucun candidat promu ; référence publique conservée.**
Sélection interne sur validation : `source_contrastive`, étape50. CE retient50 ; contraste+UL retient100. Cette sélection interne ne constitue pas une recommandation de déploiement.

| Mesure test | Parent étape10 | CE | Contraste | Contraste+UL |
|---|---:|---:|---:|---:|
| chrF++ traductions nouvelles30 | 11.14 | 11.32 | 11.35 | 13.29 |
| chrF++ répliques nouvelles6 | 10.80 | 8.31 | 7.52 | 7.82 |
| chrF++ questions pratiques3 | 14.10 | 13.87 | 13.87 | 14.28 |
| Répétition mots4, nouvelles traductions | 6.0% | 1.8% | 2.8% | 3.8% |
| Répétition mots4, anciens textes longs9 | 21.9% | 17.6% | 25.3% | 7.2% |
| Réponses exactes,40 familles déjà connues | 26/40 | 25/40 | 25/40 | 22/40 |
| Écart NLL source,28 paires traduction | 0.095 | 0.098 | 0.098 | 0.084 |

**Lecture :** l’essai UL baisse les répétitions des anciens textes longs de21,9% à7,2%, mais le score des nouvelles répliques baisse de10,80 à7,82 et les40 cas connus passent de26 à22 correspondances exactes. Le contraste retenu sur validation baisse également le score des répliques à7,52 et augmente les répétitions des longs textes à25,3%. Tous les essais donnent0 correspondance exacte sur les39 nouveaux exemples. Exactitude textuelle et chrF++ ne sont pas une validation linguistique native.

**Source française :** le diagnostic élargi porte sur28 paires de traductions et6 de dialogue, avec rotation choisie uniquement sur les identités des documents. Le contraste passe de0,095 à0,098 de marge NLL en traduction, sans gain probant de compréhension. La première mesure du rapport historique ne comptait que2 paires valides ; consulter DIAGNOSTIC-SOURCE-COMPLET.json. Sélection inchangée après ce diagnostic.

**Données et limites :** nouveaux tests issus de4 documents jamais admis auparavant.30 traductions,6 répliques de2 dialogues écrits/générés,3 questions pratiques construites ; français adapté par assistant, cibles baoulé publiées dans SMOL, sans validation native. Aucune nouvelle conversation spontanée enregistrée. Les anciens40 exemples portent sur des familles déjà apprises. Même décodage glouton, sans blocage des répétitions. Certains gains de répétition accompagnent des sorties plus courtes : longueurs, EOS, sorties vides et pertes sont dans le rapport.

**Sauvegarde :** meilleurs poids, exports et checkpoints complets de reprise300 des3 essais dans Drive. Neuf fichiers lourds,1 252 347 930 octets, aucun téléchargé sur le Mac. Journaux réels et code conservés. La voix sera une étape ultérieure ; la qualité du texte n’est pas encore suffisante.


In [ ]:
from IPython.utils.capture import capture_output
try:
 with capture_output() as capture11:
  exec("from google.colab import drive\nimport pathlib,sys,subprocess,hashlib,base64,gzip,importlib,json\nif not pathlib.Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')\nROOT11=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape11')\nROOT11.mkdir(parents=True,exist_ok=True)\nASSETS11={'mandia_stage11.py': 'H4sIAAAAAAAC/9U87XLbSHL/+RQ4py4ApCEEUJRWAhebKF76ylW217F9e6liGBQIDEmcQAAGQEm0SlV5jbxeniTdPR8ASErru1xSyY9dE5iZnp7+7p6GXr169bmJVtzzfCMu8qYqsownxuupcWbUxbaKuXgd1c2RN6dGsVxmac6Nmn/d8hzG/vjOGdxkmRFVm9qoow03kqiJzuBXmfHqLNo+pFkaVTtjWVT3UZXUzCjuc2PkOleX740yqnjeMOPnKr2DfdY8vi2LNG9qZ/DHd0bOV1EDAzVA58aK57yKGsB2WRUb48unm7cfJIa1UeTZbmLkhZHmdwARJpVFneJi45+iYptxZ/Dq1atBuimLqoFtm3WWLtif6yJn66imhyrKk2LDNjDGmnTDWcM35TLNOFvFrOJsm6dxkXA83oAwwF9xFtU17C8BR3WSxo3apimqeC3m0k8nz9XE5TaPm7TII6BcbbxRKzaAQxqFNbJojCP1+NjQBQ1dHBu6pKHLY0M/0NAPx4Y8l8Y8V2CrhvjXEfyncH5Pbz+PPjP473WRL9MVU0wZfPrlly9BPXbwX0ew9cwUcIZxsSmjuBnWo3rIm6gE6TMHn79MP35m7z4F567LPD68GJRRWvEkzIq6DupLp/M4GCR8aSCYjCPVwk1U31paIIA7S16hOLI8uLL9gbEqsiR4bLYw39KDs9RPT/O5jaJopCApBrB8xa2M5+0ke5ifevbTpOY8D2reWPYENwtm84FYxxpcyfPthjZvscB9jVUgNtVvZykChI092DhdGulPQT70DJ7V3LBsWIHQnagseZ5Yi6LIrJUBVDNWuAsiIZ/ygvbFg9Gb5qfgwsb1AHPl40QnShJrBa8q3myrnAALwm3zLL3lWbouCkFPawOCnLEUJLxmCb9L4dyAfcQWbAfEjDLk5CJq4rXVmzPJilXa1AEtt2C67SyzImrwIEQlIebfeFXUIe5p7VjS7EouB/B8tqDjLXvo05E2sn0EM7tlPjLlYWZuM+K1ObfnEkbD87qoemMSvUBiKXAx/j4wdg6w9wJelFWxqAOBvVMXy2YTPVhDz3bAwKx5BT/ZzgFd3pThJs0t13a2eQ0Wjn/jOM12Og8D4y7KtrwOhgIhgOqV1pC2EDAsgB54QxDqS7vlhyWWnSByAHC7sewzYj797Ozu2QxMoNWO2cQTwcuKlxzlamw1/KFBrt0XYFWDijvLNE+iLLOqV7N/+9c/hfNT6x/8mfmf//4fc/lsn7zqWjEnL6pNlKVwLPPDm9cmI4hOHNV8CWJmIe6rKtrUwUwINe1EWjQ+rkU0wR6e2/P21PieoNhD/IkqJR5tPP6D5bF2hi3OCHoIKFpVcV+zprjFQ1a83maN1sKK9oVx1LmH4NEUfsD0Z2NUiGpmNiQZQWACwbKdKRTufH4K8BwFf2YC0zZlA9LlpEl9OhvNmQErqxVvTL8/M8rre151Z5qwv+lXT4AB+gCyqA+WlFuJz9xm8oWEOrftH4MLbwSLxJGU6j+0giIGBoN/FPKV5tI0hah2oGpIIjCrq7yomzS2Ngd6vHE4yBpYLqHQ4TPaDKKTNmtwMQ46eOC9ZfvKa6IBhvOFOcjThjSd7eyJcsh7ow5GEcBI1xbzUN9gi2ILkkM7zobefH4qfvrwu2ctpELPgEhI0bkiXpgUMRiHHIj2u+Db84NkDvEQUV6AcMPBW0iaa78LejO+Hc4guXpg31CyvqWlJJU6iH1oZDTT0X7nO6tl4CMCr02yYcKsMRMYkiYh+jQY2FdvcJTAHXB2QFDTFyZVcWKGE+fOhkc5TS15tdk2/bmKL/tzV1Gppug5QwXY3p+9TkGP1fw3TsWzreWMTvuIDPt72e3yOi5KWG4m6ZIEtjEUj1QcKVCP0INPOhJMwRsjDwf6WCwhyCRfX/E1iAZMNp9eVgbpbGGmVgZQXskwFkdl4F2PSDF4U6VxzYptg/IAGFc8/M3VxNgymLWmF+yBdvDhIk6V+JBZAuDzCQUg/RU6xDi6QiPnbMsE/SHSNVQAQrSs4wDFBV7ZZyJkKW0mZynIR+YjJmIB/RIr+AMEzfWx6WjDXBYNFwI/0HylEDCZEYjO9gigIekI4RDlFmIDBHL3k+PR8juy0zgTYulm15uE/D4kpFMDFUqrTx2JdTsVGASiQWAQl76p3mcNWuw9eHSCFnBLwD3AXcM9FEeisEVo9Vmr4DKqqViHYACfaaL5lWLsHtEj9iz7Fp1grpVc4SMhIYNQQcRxdclhCG3+FlazBOLmGNRlx+p0lUewnmtplpkcMuyep6t1w+qGl3UgonF+x6tdcOFi/MGrals2YdQEH4ocAmup9mEJMZx4BfGXgBG4DmqX3tfZ3MJvS2QBdfCl2nLGH9K6CYtbeoJAEpAI9IIzs039hjjkgFOeLPizc3AI5wyMooRcDUKYSnoTenZukmjzJ0EdTEYgLYXz1BZ44yp494ktIAupA8u5Zs71lc3EKcKEx9EucFwM8WT+KoH+QUgUeCmwd1G+jTLIjXhijUc4FQgYiKB5CxbuLq15EmhySYHqmP8A6Cbfwi+ZFqs37oRnUYkQXGeyBpLBySHqIVqQuQqEeTbBEJr2ZAGxGubjxJABhj5EPSI2HBejozhQYWqUWDgKGW4JWUBMJi8w43JrSgrUIVpiySHp3mLwuEqKMJ7SD5i6EH0RMGaRDThlSH4tWEEDoDITzZ1js/QgzpQEdyA+FNNwhnwZVvkK54iD4Ax40c6i12qOyOJxkh4vd80ackYxQWRMQh0cSk2ABNskwngPoePv/hYhxtQABkdaIMh1CstZy3W2x3XW4TqTfGWSqUxzlGkmwi63c5EdoQ2B88MmJgSasI0ECU/tfu07vXFfnEwVd7TaSwEIM+/hUCDDYA8SfJRIwS9EKyS88EFiFhIUEhhx+laEY8rDIISlREJvPJ8cyraYuochOD7gRmbEk7LCsMicfrn5OPW88NP046e3n6dma86cHPSY7BVbZtt6LQQVFPDCqXd5bKkkFbhWNQGWcByg1DKMiy2aMwzQ0HLW0R23EtAXUo+9xaQwjx2R91sbasZU8zB9UeWR5iVerjD4QU75CJVJjxGSYYX1+A+TKuGLNR09gLWtHvitvvSndBXBV6qy0qoCM1olkIK86qkJzNDy25X01YGk28/oh8igZpD3dPTJl+q2Uupm96XVJw05IrJ+R2uelV5/X5+OS7PfVbO+ZPtS606PSMOQ5KSVfV9pZlcH/FZNj2qDr16yo2rga0VhR2TfP3iFySQEpuBqNmkcEp+EzY5BrPV7LFtaHccIUfMKksbawQGTPd76zxiSnmT+b9iVnjU5RqBjZJk/2eTKyLEGLmqpNpEqyJGRD7o9u+Mc1Tw4Mc84VVjl9nPlTVXC/Sh11pVn2ePnk32EFbgRe3xJlzXMffF5QhcHhseVkWKnckIWjXhCFkmAFfFd14NiVS1cVeDF0T01RQjJLJfOulj8GU8LmTmFDQA1pGoTPGKlAc8WWOmpZ5+NXapC/gj/kkJDgudcnTgXJ5Z3iqVvJy5qi36U6YmVDseuKtMQhvhMZUc8Q0lFyhZDirQAxWJb1n65mpkZuHaIt04IAblIZFi5DDlYWgPtoqxYbTlVPDGUtVIihCF4CLEQ5u7roqhllRBWSxuImNB5Wii08khlA6JfIgzLySblQhdYTsxnuS5tdMrOKryWaOgq6F0gkD+J+elhRH1C2xAWe/brNNDbTDo2C17TU49xIsyjN04C4SpIH9GduCbOKKo7eb9eu4/sM9SABGeDAYugAi7vHFgAdtyLzlHFgjO5QJIdUziZl9fLNAfmWHeQ6URgRIw3eABA/yMG7NOqguDZBHUdinmGlliTULwD9ONbtAJWR5pPJRnuWhJM8LhAMfxn0poseKMxQ8S62cpzstSbxMY9mTqk2Bbi5uoWaAa52W3wXEn9GPktnSidbO3uObVzOA0E7MOg6TTA7QCKukBytk2aYZ05LckYhFjEDY+lOp7DOJI9TJdoKgTdlb3oxBm8tGRUTBbi9xcuGt29aAzzw4NQDKYz88wUloGB2oNZ1bxjY11SkkMd8cbBXgzXbi9y0MA10FacepBxIHTiIb/bt//KWtvPWXdCsZU1v0VPlbu6WPU8Ab8Dm71Hhl9v3r39+ebL218+HCUG+l8n2W7K2uJ3+yfEI/K7mQkIr9JFxmXlEl8dOKsftd8Qqt56uKPTJ3+toyLyHLqqLj881z3ghn7q1gl88m+4Riw/nCBrGSJXBSWsed5NTgnp70tOn8k8CWQn/RTTXnCcnZKtCFWIKgoOPc2fjQf7Af7//7D3bxPAUnUKdDhd7qgAoa8CQEZ/nv769vV0on9J7gtbaRG3gSVNtfPbSxX0MVhe37/9nV2wS/YDu2LXzHOZ5zFvND85Z8dfk+uUALG2973ARi4beXPbDoJrACBvP/vXFrOZ66CtHTnz+ZxV/OsW7EJN8iaFFW8Gj9w9zlwG0OcT6/Ae0e65CYm3hICA5dKfXExXl6tA38dbd6A6i7AGFgbnIzBRm2B8xdY8Surgkt3eheLniIlqaRVm0Q68ReCBv+q9GLHlMg9x/eWYJVVRYsncEaqNIVgw69y5nbOr05ThlZi6O5t5I/mGLskeD25tQGsqCPOYuoHTz/I+Rj4/Pe0Hy2O8W0T/HsxI5x+YDGFgS+adM/F/2FneDgezNxFETEz8n0qR9D96Vrc+uqSLsJfpA6b7gW53sDDFh4GQIoCahUe8v1zEBFLipsudd3HroqTRmNuss8DrLICTjA/XtKeY23Om9UjLCCEYBHi9SJF0kEWbRRIZqT+TofMYZZXkcj4BfyrHN2yBTNrzK/7IYa278nEVJqgUG6nmGOcLx+6QqNr9rPyhZWMzSZOQ62qSQHbbOB/hX6tJyBci2dFtivR0uc0yykdBN8GdCZ8ntKJbbvWuO/U9+bw55JOhiuO6MM4aiFzRSZu15w3FNkPJMrMlI3POZTn8SpbCx/1KOF0uIyC6VyZkRRZ1pEQOT50KuVAd43C9L3DtH+Jvgfh3YBN1vS/sRHw4O1KKP+aCF/uLBXbfuZy2FzKLRScBCmxnlFmRduCz2zlbdJ/stqbRztLydMvuajnUKS2jG4cIwZw7pACiKC4WLEUvz12thyRKXXQaROEYPMBntmx3XwpRbutw/ard/BjsCAAA+EMonRLb/0iZ5rtLMQprxDQIEFXqXJAxUxQ3pJyiozAkUYbzgh8PhTSEi7S5B1xDOq8wIRCrKHFQVKUSJGiAilJqNROvzsIrag4JwVM0lFupQX1bRhPC1qfXZDLbiXAu9JopOB6w1uBe6g5FwN1GizRLm52aXkZJAklByIs6REVMNfnx3jTbJi1kqqmqM0KkEsZZBCFXoo0lZF0g3zu/DXlUPCQCpQ1VeXzdTdCzleYZpZ95c5Zgf+TZ+x31SZq2vuARrWBtOTetw+guSrMIDPbR0cXSuwxBhLChDywolcSprWEvEsOKr90J1MRA5wYm327CZl1hOAE+eWBQ35+499u76turW+LEM/PX6ae3b96+plRqKEv9w+mX4cfppy/Tz7KQqaJI+4V98aoe79phF6AsXjla2KdyH9Seq+6E8SeuoB9i/7zY3vFo+zDElDnLwPGJ6mlm2tRyBHHrJTbOXB7CuJQgVMGpHsqN9wBU18F3NUNem5P7Km0a7Df84WC76rq7E49hZj2M/gzh2N52CbB9dwwCvBIINzwC41wNV5SO82T4Ndo/syQGuIt7MGWozmTwsKM2jznYg9PysjMAP/I6I2s4wT6rlNd7S0X31d4y9VIeW49QZ53Mv0MRMQazhzZEI9Ddq6geauhb284fCBIIJnhaWBaqeaFCmdYTWn2AEjfRMqiO05+SFWjPZ/7ohfU4OpDtzUErdp3ddcHyTNZuGlNFcdiaso7k/TnkHuZFcu664+ur0SKJL855dH7pJT8sFqPz5fU1X15due4y8paed3nOuetF7vnVVbRYLqOLxfnFD8nCNRGVHXrqrtOWzdd/0XWwBIN+Dgwi2W5AdXRxSXfDAm/ZBXzWzhH6LO/fKGsXewtnfBjojUdK5cludYboigqHu3EgXcMb3UiwTYhOTjTK8voO0jmnKfRt4eagjKAXtO/oZll2YgyMvzPepA880c3wypkYiyi/ZZ1G9boAZdqJfnWyiuBW9hrXFzujWXOc8o3nUmCcQVvAxBg7kDaL3i3BCuTC02ImNZRLiMIDVZTtLe+1A2yCHv0niDK1Ve6pmVZWIe2nyjSAYI/np1LZZ753KaJ1kRMcbdFivTJqmmACpUgEMXqnWZJ5DLK3a+Km6BDcz9NxdbcnR5bVsD5AbbU+HqdTA5QZancPnaMiLN1DTH3HMltVRgQjAaRS23ckmkJFqoy42J3QC3VFRxCyZa5tLGqhti2HT52iBEmO+aHQ/b0ihxki1PY7CN9ICuIvasQSo69Kfd9giGPVZu/aijxuTxgY7mRP8Mp9M1nFwJkMkz6rp3CiWStGd4EhAskI1TZRduo+QHIyoeCuNhOopZ1+ThCWhkyxkKpW1AiZAUU9dTD7HvfMtHVRQ3vut50ghfRg/LccOPtNn8u+w6eyvZM0WDhUYz0KgvLxBwjqU9RnbD7AcNr0PY+Z2DBm+ubIHV0OPXfojU0tlNLs+n1nQcPyDgDyd/fq0h17l8yMgLcphs8gMWG8xlIKSLMoJ5hoRE1/PJIlTUgNqjCqNqYvusLwGs9/96nXqGAuIcBfG6LPSvdRJY7rMbya8Fi9KYoGa+uOe8EgpdiM3Ri0Iucj9/cmAyE18YshLA2BoZBfIpk+1tJcBzRP1os6t2swCFnuM4PhNhPjjjd/IuiawihiYduPSxKHDbeYIIR7lVasz24s/K/36cGevBJ8iUPnagE7c0kOSsM5N04M0UnpjE5lc6/x4d27oereHcqWWHhnMxH2GIeNs/WZKI/VZ8Jm1BOjuS8McUVpiItWQUw8v6n80ZK8k5AI4W6EHVlVwOidgd3aIMu1EUclWNuJoVLBzhc2NTbhKlt0NSTyiI8BjGiBwZRwas06asQ3KdrS0VdYMm06m/7y+QzSKbFDA8o4MRzPwJ5LYwjqZXnD0jYK8a2K/mYLdicgZGfRY6ouU5i4gEjlVhx5kz7AmZ2xq8NWb3TqjC4MqfdXp47nGui+8LVrIJYiKoQR90IYWJ6o6eOJsQCJ7RPX2NZcdS9rKzxB8yY+eEtzygeNw8/czlQBt2aGCKdcFCAp6ZJruuAmDwKe11CRqvF6SqdBtD+//+Ud4bmjt56YiUejRzFpEWURrEvkjGjbrAs8ySKtmnUS7f75Bj/zO3Uuun3YBl1/0VvRE6yZPqQeVGDOWMpTG+fK2HiCtX0w93fcGzNE4fX0xwBvGYHziBXrnedgEA9FZ9obmVDjh4tEhlw9mcirW2PBgbYcfb2BtlRSEGxdqKlPNQB5MuWL8UsXbSwfsaxcYmu7Unurk6PZtramQudLZBk5pyfs/kr4fwvQrJfch2T7w9BmvdfgWZ4bAe+oh+anGcR0Vn/CZTsuEXFWWbGwzBOn3Jm2DYdYQXi/3/8lIuEQ64yWq1rJRAVezAxDyMSxER9xQqpn4AYg1PBnZkHfgnLxTVq9QcVAEU8hYEcm7HX1kwPBTx473gjvEvlymcYpSMlOVM14+/Umsk8Ap2Ia1jPQFC22CUY7E0PeTYgpyPm489nomoNekPQCXGUdpV1sZ8mYKi7glCSyMeYR6HpBxQrqlIqroq6H+nuGdi1MBcEEEkBE1uCnJaCilOE3VDNnIgfqWrA131YpffawiXat6c34KkXiwKblGrwa4sXrElslyL7K6hcKNgad98oqaVchDos958YbNMRrDC1hI8BrqGwBaOx2AYKD3kiGjRi4I/1ykcF0klngRPpgyBwdjvzQDCkBokAGGYu0guNWYKarBEUAduFIBRQBlBiCg+RTNS+xFpbewypqWUOKUOQMIcoWYlZioDzaKirP4nX1hr7+RQ6tt0BlTWvYpcIbhxzAQDA9MI4Wmqb/8nH66e376YcvsqjUhloyH5VHxlwE2wkoS8opZ4/x3ctFB90TVLYrni0KtJ9RvgheZFx6coF5ddv3gNWu/go181hjwNcXd9rDC1txg96XZQf1GI3Vqs3ixJczwid0P50BQwlZc1BfQMa0KQTmnXF75gIK/AHSmJBCJOoFbjvZwCGe5OgCT0p0aCfkHE4K+v0VHNYJkEH0cMzxaTUzn/+0hUCrm6ygCgKPDIaFDDfSmu5v0BBZxY/UQwiGLUvCmECDe8W5uRrplZN6U8ruYqKtHqcLaF03P7ho67SltPdt6hc72NLPWX8Pv2QkNvQ7Ywp7vyC1It0Pv0b06ivTaXFC16+L7idQpr9ipoyMINugzxupabrTlyeCqAB4h1dNEFmgqSiWDUQ7aPFWBUzG67yMP6BFBEFLfz/u3BvMLBWw4W0tsFncRzJLRmLsCqLmC/XyWDWCwTLPFbehMIU0Qk0R2M0JyDNTtNyPsclNzgEEMaQLHp8G+naSWfE9297b9KWythszkbr0LpT2yyq6OUjUbhDYRHeSB/JPDjgimrA6bUOPZrsPSID+jRkcJGQI54nVRdWEt3wnPq2xVcXXtp01f0jSFRhZaumSfAz616HK3r34ndD98e+C+peKVLRobxWBUEd6Zjt9yrrQyR51L/1BdY51y27+Zq9t6aDs6L9QkRQFFuDVDOk2h9SaKJFSD7ygDTX17zVwmj4c39TnMv3t/dML3deiUAMiBVyRPqbd8y+stNAVYIqOnK4B/y/c/GHS/BjNWtLNZ4Sh6C+P6MIVjuuIr9st+8kG68poim7ywq/acRIDoVUtB7c+Uel23gPd7bXC8t5Rn/55+m76mm6Obn69+fBl+GX6WXl3ZVvx479dr1FfoKK7th5v/btjO1PLl7pO1oeTmv4km+VeWC8wfwEAhkB0VQippbonRM5jPSXYv7t5oYak7mxEohoAPt2rElGeeeZGpP9tAE3t3Ysw+U7GfsLN6LfaoUQJ2goKvQQS7ReOKrYL4SRh/2YIU7eOCd6bi/a9P1Fa/M68W4gf87Eb0I2XmiiDOXwAS5dFC54Jwyw+nASzPlEfn6KFlwwittDf1RAH6LYI4HfDF94IiXhL/RoHiMrmD+8a+22w7E8fFv/298QTiRQIf4D1L8i1VJDW+0CfmT3HzHN9BlyJn4Lq8+nv4g+5Oe+HdOKkx9jejQOPg+0LxIuA96YeA41BCmkDCtGxIFdw9qXwVsNqQ5b5iwHnYXWcVGzZoXQ9fCTxeZImRZK8vQASew4Or1KoQBFIEZR1WvOv8ACqMQotx7Hgouze4un4outfv7OL9vDqa//SC3Eve15UsUWdE1/+hWcUvjJ4PtYRlMRm2yV12opSMXmQ37Tv6k8pyHqB8GehrBeE1FKQtg0cqpLRje7ku/nT8daFKqLuCfWXjKSUiDMhA9MMbN2sbOs+VBiqZCXGpO/rSuzFwJmyScMMw3JH9AlDU/2tnxKLOJAuU/aBZRRqNIdoL6kxa7dMGjcFBDH4u8D8fPPHX6d/uPn083QomiymU4Hi/Dlv+twCcKaKXCFewFFvThJSL4qiH53W9GePJpaiRNHosCyGfx5isWtwZkkBnWU76CwhCAGG7l1k2E8t6WrRpkK72BTPyYb89zcffn57E6q+/C/TT+/ffpiarBdKK1H5mzn/A7lcNngZIsGRWjy7+olBWIW3L1Edp2kgE5PeJwL677+gKA1ATEIqyoUhuJ8wRO8N0uGLlqHBfwFcjbleyk0AAA==', 'nouveaux-tests.jsonl': 'H4sIAAAAAAAC/+1dTZPbRnq+51e0dGFSRcma4XzaB5dle1NyYjlb9q7LFaemmkCD7CGAptEAaWorJ/+E6KbLVuWy40MOqdxyZOUw+6vyPG+DHIoiZijZ9Mxq4JIlEuhuAm/3+/T73X96aOOHH6qHfm/vUVIYPzwrC537VJfW5Y/2Th49edjF3XFqSzYrjS95odR+JN/rxoYXY5dpm/PyGyPx9rhw2VhGeWp0FblqrGKjBib3yhuVzv8yMXmp9ETj7xQXzMQUKq6Ud6mxqRq7qlCRyzKTR7iRGnw9x7V8fmE4EH5tom36mD+F352agj/1NZ5BjfRIq8yoc+29UzmuTKxWU60SWxo1unw5qnLVN6rv+Pd5laFFxq8ymMePROYsdlGF35YXGOisb/VZORvbSKdnZ1mSjF2ZlTrKnC1B0atuic5sOnvLTjaPbWQ8ev3rk3+7umzyQWr9kJcffqnzmRobNwappnpkFMjZN4krjPJVXliQtHTKl7ooVTk0FqTUs45XU1eMHj9cGbSvXZUaGfOXEuvNRz1bTnpepenV7apISZJhWY79hx98MKwGA5sPEh2Zx5H7INZYX6b0HwycG6TmA5+5dIU6hZlYz0XFlRbrJOnpPf1k7zQ5OU1074k+PNrXJ4fxcbQX98zp3uGTg4ODle5+qPcPj6Qzmhr86InZi5PeQWKiXv9k7/i0d3R8HB8kvZ7uJaf7URLtnRyb0/3j04M4Mfh+Gu+ZoySWGUsxUbkn/R5++umjp989OngsLAMWyKPhmSvsIPAEyGkxHVjcS6aJP9CxHuNfFR5NfV5PMPqHaVnpb37UUanGVZ8t0AUzOzDl3/t/ABfY3MQfqdyRm0zBkcEuHotqYtTTML8YMte8IOQzUxOf9WdnmLzYagyf6NQbck6FxZKXNkKzyBUxmqFB6gaVWWlVU7J0I7w7rveeCCbwga4u7p38+9/96SZ42dsBvPyz8UoPChtVaQmU8GCTSrDFL1ClBpmOrrCEBVgwmC2MoIpXEx0NMYYD9nRcFFVjAtHiJket8ABrQPM7a7yvVFJdvlrhns4gvXyVVfgQX77KqxX2Kf/6U2Vy8lcnt+yRuxx/lM5fYAQ0yK2Whok1vJ3akb4DcLTXAEe/00VmQJ1Mz5SAT41FsZ7mBCL86qjGochNQVuQ3OQxb9VXCzf2TcD02xC3ha/7Cl+HG+Crd7wFfO3vCL7G858BQavYNcZGDjYx3GyLRilpDZS+qEQuCnxTccFPL1+qc9PJX+AOWGZte78DALPfBDBcNsCYXCBm4IDOpUg4Ri+wBlNP2YdCIkhYjZvQ5O2p0iLDPUWG/d67Cja9XSEDdllsZvj0Q2UXSk9KhDBctPi4hAxdYWauhJ5y/j9lEHbGmMJYpB3Vr0TOSYUjbK4MumEMIEx1nUaFvVeUhNjWGsAoBQvFogpwp17ZpMPezL1W89fwRGEX5s8seY9j3AHw6TWAz7/UetbQif6kQCiCTWTLmcARVo9bamFGF+mM0BTpMhpKQ6FyUZO4dDcrYbuhbwtj91XAOdok4DzZAsYOdgBjf8ixWJ31xKX5he/iXx/sQASmnNA1mF/k84tCp4YooLCpj01pSxXPL85NBRoqbzOy5IeimGHluwrYZ/HykIXGZDOgGj6Ww/nFGoD9E5jDF8FqIUzUFZUBswGesbgpzNcX5qt1DDW7fJVSc9Aj3VXjkc6HVd6tGe1OiE0HDcj1VR6ZhZlIF8SnLhEJ+BTGB1IZDcxf0LNfGD1KtC+VS7BGi8LGAxCVl+Nu0Np0E3LtgrAtZN1TyDrY2wBZ+6dbQNbhDiDr89xX0AS6yqZirR4X8wtoZKKYUaICAi1s0Wto8wxzJKYGbw3+4NMUaFb+9SdcJVPcAew4bMCObzBtASwwCYoIMANuFyLRxHrWBAM3v3HL1PeUqfc2ySF7+1sw9dFOmFr9IxmG+9u6RwpbX0nujiGaTOY/lyKHeE5JbDlSblIILoAAZX40snNSDnE53t+rwvWD+fi8GlNlE4TAh8RkEN+7VLI47hiTDJLh19hROvByNDSZ9av9hvR6+UbPVngJRQ0BO+8UWJTO/5OPip21q0bYSYvLl33IQpbmUOh4ObWK8dUFGkFSHQyhaF3bUeXxVsymETjz8iU280W3GZZqtex2B4DsaAtfGZ4buBZoptUUytrVpOpURakrh4CYrvIVlDcd5kZBW/acEqCfH9kCK4FAOHW0R4E1QiMSLDQZLpugwc3etrsygS0u31NcPtrkvzvYRtg63gEuP0v9mg1L0LigGpiAGLoAwYPQVRKaCZzQCVWk/Q+VKct1EeypUSP0UlP+1YceqdJqVNVMMQImGHDLHYCv42Y5bNXMRMjSaghxzEe6SAg4kW40dm/58i3rtxbuFZGstwXrn+yA9T+BYvUXGobW7T1diEU6AgBMNHhczf+suB2DrHZiy/kFreEOjGVpP/IbDD5QR0C6pdlB7BHdpbWCNtLVYBZf3QE4OGmAg08SYuHSUNNdSDYDp3RfvGJ1uA9+fUEha/x1Vpu3p06LF/cVL07f1SN2uiOPmMkTEbwnDgQHMKSd+UXkUjx8yVcmNBhRo3TMgKC6YaN/6ykk8uWy99UorboLgXo0huA86Fu5PbpDfqvTBqj4dGhTqC45sYG+8mjoHFhzissmkMPXt67zSL0DRVp4uK/wcLIJHg63iQR88hubeFZ957GmFaaziBqsCnSAvBzCcBYOcryq9ZnZ0g4TW0aX0VkrPBF8IYXGN9O9CkSbgftk2Lsgcew9eXsDSu0Of41yCfqJDYXG4kC1tzWCvD31Wsi5r87t/U2eom00mL1dRB9/1sFroMlGfIkzm2MiCulHx3fIaCD5CTIdzqYd5GJtXsOZP4ZQ1WVEyKCaXL4qQlMs8C6DWLGpl+CMmaNPldcCp8ieje93AWKaAoi/wtQXfoknAzcxRc4H7DKqBWvMe8ETE1eRUK8JUX4VOrVgcl/ll00eqv1tPFR7O4kF7mixiDzKbGxXA2Vq82giWQs0fo515U2NMyYkNBB9ghOJd+YXZYj6o5vJ+qgqJY7YRCpeINYytnCDAYUPoMCHV2aBc/5bR6fx/6ARkPWcqjJrFm4HagZu5NSM7oy7AEFNIcbPQlSfpn0ld8TnWuSh7bVkdJ8OZheVYIICVEF1KkyqfxS3j4uwwOwLc52p5VejYwtR91Xe2aRi7e9tA1G9HXtrJDKGTlEabOsgYy6fUiy2IX8K3OvpAa8vKGFnWmvoL3eVKRq8N7Qu5JhRwOAie+dcd/IJo/jpENXT1DEpsQ7rX9ESRiGg1ml1N7Ss3k1+noET3zETENJRN1AQmAMoiL24m8MjiDw0TtlBZ80G3l+bgC3w3Ffg2RiSt1WW58FObL8gqC1WpKJ4/l8SPYwHn9QZETW6rGHK8058+VJiHyj8LzZibMyiG4AXsOjX2OEuAMfB9YKLwUuL2aUWWxwEmVzCe2ObM2Z6asth7R2SZ7nGK/ROFGqRoY3rW7m4TfzI3uFukEGgoFAYbkMuQWHG2i8i3iP0cn5+QbtwYV90JZBvfjGomEQokXrMBczjoDgxXdx67/I1RPnmKsqdH2pvap9yft65/G+NSXeyqXKnDdFXYXs9r+5C+sBeUwzwZwE5ILJdJQwskwUyo1PmCRQYBmKKwfTrfsoIRwopGZEHEgqtwU0w88vI1uJNqwKtSCIHN+FN72An5Wy+6MwvSg0WYVDas5zWA1N2hew/MCsp4I4UkgDMQO8hsmAZ0XYDXvmhIrecd7RVkxDIJkSPGNcSDDq8lph8/nMp1V2KARAIE2skBTS2XkM9oJM7czkYDpu9Zt0LJolbwbuJLSr/Zl4C9AEqA88707RKrbL1k1v8BNlvxqoNqn/5sk+XTNj5O1mXKTu4O61y9pFRFs9rmKYo+TyS6ePNSD7ygjyC5TD1A16+KsjRzZE2pQPenPVOTs/OvO17i//8rLRYPZtgb5vWW1TV+f7h166rnkHx8oAYZkEF0couZzXjKPlAZTPlpvmKbXrKbIlnyuspelhGb48fVWOlwTlDACEmKcbtIb7rQA2OjVG4/9AWd006xV2ZqBZv7yneHh5vwNvD3hZ4uwsP2xcmRKGByWIA3Ifq95Wz6mP1jIIcuB+SyYN1fc8bMMGIUW2dzvPOi+pj9XlpaXL11YNO5zYgqMkRFsCHPNxV3z/8FtjxsXrO7PLZg+8fNmprN7xfy7n3VTM72CQpPdmCc3fhzvqSUpJIOlFqf6hE6fIeW+FHyrBcQdbRkPQnboY7E7qDIbyI9ZiCzlgXmoKPZd2XSgzICb+4JMGeZlIqa2sC0TnTsib8UchSUOqiISlRqIi5WoPQKzjIqvG4sJnUFdwoLn2t84Fk1rzCNlzaNOy9WKKp7JWMjFvZlmWHreoyDqUwphqNsdHqZeScWXAsBym1CjtwaVLz15/4kPWOrceYgeDwlrI04iUSa+5CrrO3IkA1+dCehvhjSkKY4miE9Y4nt2VQCWXq3IjwLYW/MDFT0xdyiZCU4iZ6FEb0zDCxOl32FRlKGj5TuTGxkfphMeSw1ImcRRncJeWUme64k9gfRXqrJ7QhAukuz2yL2/dV4tqk4R4eboHbu/DxfZI6gDHQNCV2C+oCbBffbY5ZSAGvgHGbKkK4ZMsvqrUGpbdgmqwt1pXU+AqKWT0wnRhozIznbtBXRd9J7ZLb8k7980xiVLOrGyyIgytB1YIWw4AfDJFVLJQzW9dwbgVEmxyBXzsAXGzjAHzPFgQ28RJIBWKJiKzviusJi58ErXShTwZrP7oFXARpCYs3aZk7I2+LZPe1kMcmCbS3ja3uYJe2uqQqrKl+fFNTzDQxiW4tqx/cBiocXKsYegdWjh80KoKvP37LdfdV79uU0X2yBdMd7oTpICVA0dOz4GZLJRzRL3aWooslDZaE7JCHcrwLPY9uOjGTv+G/5yZVS9azsC/5IMdPWdfAXZOWtVPePWzk3RJwI+pKldfvLZt5H/oSXnSKFRFDSQBpSpWZZq/8Nu/dMn0bvPw2yVfg+qNdcX0CCbB2gMXz/6XSG0o5rGkB0dC8EM+8lNUK1z6SYluu7ikmH5NwbRZUPtByiR/qW7Cgm76hMpDAIcIf7LJQ6NfdGsrWhaqkaXr5srtSSHxmxd+C/sxzrIJ3mmFwK22u1J/6MTBilVwT67xTBDpqRCCZinQmLihaXxjQvKov4JqW0AjLCjZuvNQ0ll2mdhyqLE+HjglxTPKSRoVZYNpyJhrUjFufkhYc76sesqnQxcHxFuC4ixo3nwY9JLjnCxrEdcXCX3o9/Pnz5Y6f1nEyVLT1rWBLU4GaZ6WoJhoYAbYDYw6GZXaNsaHxnVrmbN1UVxdPt+DNXRSh+aOz6fzP4hj6gZ7lc9q66ENyFYt8MrOTRWp0yQqBNN+H0xig2IhXi8WJ68J9V4E8PkTyxFIFMETy+NVQnjdkl1JKsGA3xLRrcUQAKdajOCQYV3zhrOl2WyLHSaMZsyveGp46NZypZx1WlpnqmRgxIuBDUqV1yRrxFVGaADWXgTPNUsS7EqfFl/saMHj8TjniAJhdVK35KqehQywc51pMHxFd3B36wOvyNJusHp+wHIKXqI4QhR/KaU81FP9bYfymsjPfOUjncubEKHdTML8Wp/CsU7BKFdlc3LZN7H39e7Y8fF9lhMN3Kh7MILSdlJYRZqWQAPGd9o0IfGtyVl4zvsGnyRSCiaPFYuI8EySvqj1cZV+nviEDchaiEXK7CNIsuQteZelJmUddV2kKJ4N0slsBhsZCMpLiGNHYGU5fE6emOCpr/2SoGXPVypfMMZjJqTVCH3yzeeKK7NoiEL+QZC3GtNXt3qb8AzFmF5GunxqsT4gKE7yNmAmycBScl+JUBtSSjKQ60zpiAhNL4K57SxgGKiHeeubqOPHbgYXGmFfqCBJvnzs6Q4qQp8jXaaxKt/mtWs69r5y7KTl5G+veTuq2fI0Nx1WU5Wfi2Hgtp8dXfmyi0lyf3NPF1SIzLx6li3w7YX5TEhCCoeGFQEEIoV2PVyVfbNKKeX6YHtggVLMFxWzsklJq/ipgEX/j42xxmBgDFKFEq1tK12ku2ZJQNlCia3jDAi1raTeMKPVCcbw+ZqQbYCZKnTd0vNJ7ETt6XsUEEYJVG0NFd0fUFrdat8RqpYVtgGsXkZ6flwCbgF15h8X1vWBMZIqSh6oxqD6SuguMxs7FqjlgAIfuQ1LPgVbS9QqvNsFSqhMq99TyJUq6tDz5a3Hol+j9o7HuCn9gaxcbwK2JLE0xmp8wJF3Qh1YNASDKLgBwukcNq9lRleE1S+nG68TU4BPwJnY3Ic0706kFk7ZCy011GKaFpf8Cw4/T2aP9/RvixqXZJiR5bZw17YUGkR9LHtL4GQvOeS4WhbfiufCMvcRLusz1LShJoAml7kQgAj3+7z+qiRwgT4AxlTePv8+/lH2WA35TMVnI00rApGrISIWr8FMPeCdEjaWpHP0eSuTpibNygIiOIhtTkFoL/PxO84Syrnq+PH2VldckuTanoUF9Gb7zbLOw/aeOVsxF89HlSzIkgxJisKeFEPD4hsTCw5Pjs7Opmdmpze1Ua1faqcmns2twa4suW8SXf/+QvhjvimLWDZHmBKTMUB5xj1fTn8tiVptreQScBo3zRT5z6qaPMjeR20UVjR43ZifuiLTXoRzfkVboqOJ7MZPogeJ3nWYOr5XpWFxDUaH9cC2tsoXE9yuRZpNFp7cdJB7+rUGirpgiybMNMhCe/mes8rKUbLicGs4A8oh+E/yed8B1IRarX4UEN89EjsKSOW4Vww6bMYzApVj1NJuxDDhTRQe5Tq9Nk77mNW/Ck28NJSeqm8E0PdQT1hFWqeFxuXiOONipmx+lhZb3C1r2tnZIvQEtx39T0PKcxXBLOnWZ8RdUQpbxFGmr7BiW3rtgZC9EMcoNIe0akw4Y2oA1FQuF0uciBQs2cyU2fQZ/3gkMOr5WjipNyL0D81OaEiB4C0R6B2LchFTPXcCpINgFyUfEuXElDreCAXqKcTeQuJKC8QYuwU7RwtW9O+RxcymI1+HqaOeSUHh2hqLTrK0SG5U2qbGrEsGlNn2THUwxkcThYDzHzaovsXcum1/ENtSKQKNwxi79xyOoiSwAsIpoX1W2zjo6Z4CJtsHM5TISGyjGoyatJONQZso5wJsao+pbHk0yM3VpumwRjD4TTA3aSy6ldpmYo76rg9JKx5KZgI6+WZRtmr1WLqAfBLbLlzepkHtH+2dn6FXOfL8srwG8xoZbiVpfs8wqw/sBE1IuI1CVZRZkOgApevpYfWtUYUB+KTJfQPny2HFYxgPaXMS6D1i2mfzlTTox/jqV8beh7Y06JAu0MUdrqSWT3ViaGS/82gppsfN9xs7j3jtqkcDOo/cNO790ubcYpsuIaI9ZWTb1TorAFxNbd4PkWNFDyWngQefzP0vz8FA+5ILiq0AGZNRH2fznDSedhwxHiEbB5r1k4ge/JToeNaLj04APYtWXgMch/q8hr1n0u/atbsKl3waQWxB7r0Ds9F311aOd66u3AGIL2U8CsWsvZ2VZzbN4Q2Wlg82E8EAKDhA2WAuqsJKeaEdSsyn44UI+U3x15ya/5a+KUc2K6jchD4PuSImlLFw/NRkRgu5LTeKZeCa07EorpoKCESpwQdksp/1SytyEc28BrS1YvV/aam/rU274A67AsDqegHRbBHU14tXaUK8j1h8awrgmAk5AIN9x1aSwIf/r9Qz3x+r3FU90sv7R+cIn+XEbstWGbLXQ9H5EWfS2hqbeDqDpi9cFGS9Jq2Na/VdTUiMDYew1CKMIVceMC0DRJ+6YlcLklkdltRGj2vitNn6rRZZbjt/ahCxPdoAsQIV0gRFXB/2Ni/mFNyyLC1B5Pc/94zZ5rU1ea9HjNhJkjxuS1/4fagKSm+WtAAA=', 'AUDIT-NOUVEAUX-TESTS.json': 'H4sIAAAAAAAC/21TwW7bMAy99yuEnDOsiVO7WW877LRhhx2HQWAkOiFgS4ZIxzWK/vsoOekarDkYkfjI9/hIvdwZs0px4tUXU+3X+eTiGCSfX/Sk5zYhn6wkCNyBUAwZer9+H5wSiWCwCYdu1nB9E4VRTjGht+DP5DCna/i1kHEck0Proxt7XGh/L7lH6A8EVuaBHHTW9m07ROkFXB9JyK8uHBIVYDf11loQkJkPIrex6nFvLdOBSX88CxHcAh4eG2snnGmiQBNAFJowTPNKQX8+lGkZtV0KdkgUk35xgNwhY2mhhY6xJAqkI/6HVzMp3OKQxYYodmQt014xFI5W/zN26C7WSxqXFK/tWj7B9qHW61VTbQ6A+2oD9Wa3OWyq/bb1m8d6izvvocV7Xx/apvK12/mqQazbprnfItQOm7pxi5+rjnp6P4XLZD8fMWACQW9+/fj53Sx+rI0qNjzEIBAwjmwSupi8olwMZ0xc9oWvbn9LGNzJADOxZsin62Y8ma8Qxw41bSDNxmdw0s25vgla44xa+Uw4XSsxPRtPyiYm7xwhmzbF3sTQzUamaC66L0LN2+DWJkXwmsvDKGggeNNSsRY6Ze/Rz2+7cUqYFfUZmcyyvGZQmkE4N8g6CpctKdRFs7k+E73tY4hdPI5Ze/aIkZ+KYblr1UJOZUBBvFH+e2RF2oWz1NcWqp2J6moHg5lIThRMXpvS9LUCdJ1KVK90GmpFPDCms4rJQDa6oyrmqHo4U5TLZYjlpS4XZevvXu/+ArQK9UUbBAAA', 'METHODE-ET-SOURCES.txt': 'H4sIAAAAAAAC/21XS3IbRxLd4xS1oxQCG1/RpLSSIZrhCIpyiHR4ZlnoLjSKqq7qqQ/MmdUcYg7gnUU5Yi7gnXGTOcm8zOputiwvKKGB+mS+9/Jl9jtpKy03rmllGcX//v0fcXyMslViseCn1rvoSmcUPh0flfWq1iHio5AHaaNQNnp5/K9VDT5NFmvhyui2XonlfHk2mbzf3qsy6p14JbCnShq/GBXoocVFOmpng1BReGV3zpfKCyNFhV+VraQtlZBJ7Ly0x1+lDmLnkre6EHdSG4SkbbmXtkZcOH85F+dnc7FenIlWetkcP0WvQiF+kDg7YkFObDGfitbpKohwfDQUnLP2+BjE8ZfFfF6IaylwIL7xkn4SFaItCR6PAChYHBqVqI1LETvVVASJb7fGlbJWvPzL5KwW0hhna4ZISA84dKmVKSaTO++Q1dbLgI2r+byLMSDazaV4jX9elI4QpisDkgcgz4oVJShq71Krnv/1qhfJGv1RGb13rnpWLJ4X4t3xc6NEg6QBGcJu6DkI42IYHloZArIIQP1BGy1BF7PT/awtcpJGB4YG2F4rEY+PjdNWlNKUCZwcH2tpcqpEdKt8/NN5rbMgmNQUOJF/4cEVYqNGYgPUJivFdQoKSFQbYU8Ioq3RkYIFaiIkgO218xqhQI9g6/h7FNXJF9IE2Ncn/VmihwsfXdu6oEh1B2JYNVvDuqv4O6ISmUKGB1XSF1vivKdiECbS/kWUijTJC5Ml4UJ//coqEdBIoZHhHwnPerdD1IRS5cpEEc6QfdPGWX8lGJO+VsWSYIaMCFjGoVXpQBvxjbSEPbT7me8CFahJjQsYf5Zt1FvASOkjeGnp8u6ogxKQcq92KqIfby5RYjrTQiWfS6dW5vgInavUEYtkW9KxZ7nnFIO4+/Dm+5scbMmXgumg6R5IXLqEQ3Lx4IQ2gUJSQLcekRoqGSqYSp2L6D4q+nh8vMdlssV1CY9UaSCB5Yh7EQL9R+chCRImiypocBHPT2u4ANbZE94PKXSa6U5BypkD8NNHCrt483b27fvb2SX+mMLQBxNwSamh5HyleihNQvybE0KRGa9kG7NrGN2QHJn4oAA6XTKuSVR4x0agLfgw5eC8ggyqxLaEK2DKfEYrW638lGoRHhmltl/5DPCr8ZHqdEDZhZDpFyP2FeBOB6/4vggbIxfA3y7FROluXSrpmZ2P2kMlIwnoNkv5lfiJhF5+pGCkKabiRiUvjbhTD1FcKas65/xZx734cZzzHSrMaluLZ8v54uJ5MdnH2IZXs5n0D/pQOF/P5DbMFhfz82K+Xi0u/mLBPjZmtOKwnFyrDh5uUQGJZxvuNBHJ3JnCGeTRUdEil1FH4pbTFYIKJS0pxG2GOCQDhgLhBmpy6RFyzDjZKawnJzzYwWkv99fCupQlq0LyRBNigBhRqdmUBlA3Ug+AXpG1n/5MvrLpnQrVeq2kZ/jQKXvQ38KPXQ1aR8AD3eV8jG4JS4t7h4X/ZAzp92KnQa2tw6lqrGmLb+azEZKoadQWefiJtpHs4TNZKgoJ9VL5zua4+nVtqSZyBe24XTx5Fey46iIsxI0jl9rj0m5/Z47oEU3iasmVNOoDTz457c0A8VCFoAFwIVs+NPfNs+nAzBd1hAKqnyCtEnALEo5DWKI903pLzEqKAlvBcYWSRERgaON8m6gjXzlXI6Tbd++vp2KzOf3276frAtMEyDzowAKo5G63kgsJde/OL3ZyNZcvz5by/GX1TbmoVupi8XK+Xq+fqNmnukYUOwnBlW6GSpNBxTCr+a5ZaJyZsPvn8B5WFwLlGkO2IB0CuaJaD00kiHvZ0KQE+tgOe7vDGfeKaPKd/kI/a6x58inEd8OUBcERvjP2MwDC3eAEowEmP5r5KtUmLMNXtja04fXg752vw2Fa+i8bGEt9Nc/nMiNhesbuZcj4AufBhrMc1ELhlWjpQTyrdChTYICppXNndY3bUoIxzyehPH5iX2voQIH5igrm+XQl6Pzsj63PPkvdxmKERYYk3+SHgZS6bnhqTdA7tNg463KFaZpQm3wIjZU8fTAxxuRTD8p3oxGaBQgCoHTOaGpWYdrPLQd4U5UL1srce4jpQxpu5i7XMzuYeufYqKy7y9s7YE+XP6U5jpIbecRAR6QlnnqUTd0so8KYEPo9X0j6ImMapmPGaBTtTj9Qreb5P6vxdSemOXPjuQ9TG5ap/MomYaylA4sVbcEYD5mwRC//9sPlh+/fXd7cFfdQNwqumx2/wGra+UelvnKPKbXk4yfeMG6N9Nxgyl1TImuqnG5ID8of8gxyJX2lTnfk1f3Ayq6nVeomYNIMlg/vBHjTGQk4aLgJk5JHzqfJBrMkfDSs52ilXdWyPcEhcXLo4d3JRvcqsqlXCa3kxsH5jgU2vKMYsqgP8GUMQ/Dtcu+/e/Fi2mexuUR3vr7uoQLuHpyq2QizPEVBd7qia6Vnq8f0MxX00pKAwJQMmS7Iua+7wSr0wFI2g98U4g2RzgHzK0OlSxlxCndQrwCU5FlI3Kfulag3DyqDHdpEnkyQLgYspP7HbzwfUuPhGZ33/PH7eAqlcqHxVNQYMDLwWZg0q4M2Hpc17eibfK76p0b1p9eFbnW5xzSseRbuJyHqCHtMP3iHJLvlyRWOLd5Usvlp9uHmit+XFLFJOedXTQJjL8ma+F0vSJhGDc2RDcuYXYuv5Up46wc3GL3Jgo7kq5yS62YQUAjgcLSv6ahsZnhvKPNu+OAe3ZzvNyf9+y8FSMk3rjp+Mr3jlYF7Q4mZiPtIbzed8InZyf8BuQKYJS0QAAA=', 'preparer-tests.py': 'H4sIAAAAAAAC/51Z3XLbxhW+11Ns3ZmCrEFK1I/jSMVkbMdJ7Ymtdmwn7TAczhJYkisDWGh3QVH2+Cn6Ar2MctGr3vWS0/fqd84CJEXJnkwvbJLA7vk/3/l2pYvKWC8unCnjSvp5rifxXDr+tCquS52aTGXSyzg1ea5Sr03p9qbWFMKb96rUH5R1Qgcxb9sne+dJI63/F3x2xuOpztV43O1b5Uy+UJ1uv5JWlf7saXLefHXDwWjPyqvk6X5kal/V3u0Xssy07CkvKzU42HemtqnqucLkmUl7quxNUt0n6/PoDE9cMqQf/dzIzHWWXTE1ViyFLgXkQrfMxl4tPbS7Ktc+16Vyne5oD64ka+P75B0b3HHedu5ak5qikqnvVfUk1+n+Og5sSNTtIjyDx8nwwVMl69TUlciUmKnSCadEvvplAV+FXEj8n+OBWigrslogLkrnooKLAhoKVaZ4kSv8vMCzcnWjSJC32Krz/oP4wQ/KCTmzOq1zj2VOVKpm4a4V22iJZD1RQTK2a6tYrBMLmc4hw0B5ZNK0rsiS9iVJrZGuVlO1+hWrt9UgbV5boUpRbHTd8Wi9H4ViSgRcXNa6dSMnORCwoK9rwbJ2Tm+88Kt/+2B9hTrJ2HwxqdnwnCOCBCtsgwzYUW/H6F2pUATakajVjYvx6UIySFZJ2marGwTXylwV9LMuYYfXXmSrmwtVlxDpUN9QdMrBkWJiapirVUxOVqQchuCrn69uSOfz0tXa47XOOeeVXd2EEg9eQESb0bBcfC+LCcmb7FYMtRXty2D1YvWrZxMdOZhpasVS5fAJwoVaKjaSTDTlrMYGayYhuxfIrGt048tUobqwTQW5FfImscfxRt5Aj5HsQrvtfXOqSq6HF7nbSRYbail4U1Pn0mbCBU89WU0yEUmRSndZK+/Z7yeIyy+UmN14x9AtU4haSKwWq38KMgMdpxfaI4koION1pil/6/JU5VRSaBYGNmFLHq1uAFkIqqeupU2KvZAZtUuz8HaxfCkT2wWbSQpW1PYeuiQmV0OHtFWJqGlXsKffogE9WuleIVmhSw2gkZRPqs/Q/cA5lhRBudKzkjPPzkaSA9crEILtem7yMeUGp2hXSI1q9KjQ+6Q9JJTerG586CdKuXZp7bmPVSqy1uJ11+6mnevawCjKV9OWC+1Q9xx9BgqJAqIqax7AtIKcj7kmTQ3E5OQBKLTd8iRb/YsbE2i/aPCh2dms5wUoeXdf81pVSde2LEG1casbSqPVH2Juj9XNrC6UC/UPo1EfIQSEkdrRAHkwAogfHQPEX6KOvIQfVM8vSkSoVD6mWJaXBANBL6Mn1MBTkiynnAX02GUN6eIiklosQg+EqUGFHFJDz6aqRJsgglAygwWoZMUwiaxItAbVbmGAMzbTpSSwJ7jV7O9C25qz8xJrSAJsBTgArf5aGy2+ES8oBE5PEI7fYdkrcoYNSnN9WXNsnAOanglFIFxEEhFfmGu8WdBwhw5OK9kDpJFkH+SjwDmzU/phplP4pHKK6Y7dF4RJC1IKlxF79LadIXspAdUs7AoVWVeV1QUPvY1XT3IDGyAkJ5NZGXS0v3XpgF2QCutR2WQ5Y287QUNKLEEjJtVOCLONBU4UErQE+SQQ4Wi2eZ+ij1W95Ni9JJ2IlrwOJZVzE7nGCuqEgrbAkpKTkbfBopLkkui3UqawoimebPUfIasGQXdsBKB84KrmERKenfFgMc1OzoNCwRFdQGiwcm2P+EmXmblih54Fh0LNWEo/EA4YL7kLfzQ6R6QoEZdUNgjwlHNmappVBF0E1tLTOAJQNeMeMeAqoiHZTIlNfbtQ4BmPnFDgbrvC2azzkmLFQbqQHL2USi2iWmxAeh245/yMbIT9FLkUr8G9LOu4P5fU2AtDsVgYR0C1QdMNuuUhQgogDmMWwD4OURFYjmNwVyCLDBEN2KWEKDTOaOsbtLepyd5rztetNkdzVWDO6sv9HuOpLdSHHpQF9GL1yGjZBvkDGxPaleMBjGwUlxHNWccLUAieWAl1X8roinWcCVnOqEjlxCFsUMVbN8oI9izKJRmO9jI1xajMOjrWy5iQvvKxl+59nBlkqYxVOcu1m4c3yWtTqu7pnsgSYuFDPTqTSSSi/oXRZScbRt7OXDQaXoyYkV8QI9fL7p6wSaZT39FZMo3cYND7GKR/6n3U9G85PBh9imIm60kE/3zERiRbliSNQY0ljalw90rZRMbhyDCGWTW5mMAYnUWj9jkPluvdp+ganSqXwPXmSeNuMsRKZ9O7zqz3TiQIkOKF93o92hE5bgy/FdB2TW3zJJp7X7nT/f15PZvpckad3k/NPp3KnMKxZGbMLFf7dCqK2o1W0TA2ZRJNM0yjIzmQB4Ovp4+/nsqjA3ny6FA+Psm+SgfZkfp6cHJwfHy83urm8vDkUdIcA/vhZ2d9gJpcezo0dftztcz0DDnpdGOUOnVWEj171nv6995x/yCKp2iUdD42VsPqJJKYQYAllDTNTpcDy7J9mckKnyKoFs9DDKI4BHG9Vy3B/kJLuTmWexohvuO6ggpMZWeiNMQdFCFkBoTAwNILJZ6ymCgGmOInB0VdKfJhHM5yyXcydypGw88J11IsSYG+WIKXAJRahRVUqkh8CBAf+JDYBDyugx99+IlTciekrtvXmes+PDzDhmDnZzfIZu1gTyA66Fog17JzV1F8V1T3T8nJ4BBWEbpjeACTOhYnT9SZRuSp1IadweOYjqLduHN0HBOf6Y5O16rIlqntJmxU07VtycJdrti4IkGqROtQZGn9aQAFFHVcxdE6lVEcId+o5vaJpqPw3u/FeZlf42SkiG8CxlOCs6XvgTSXfGig+4RTSh+GFrg0RgwjHJZmQGtX4bUSxKGw5M84kZSzK3ktdEbp8td99vgyvgAeBZ8fvK2JwgAFDTMyoKnFhAT3EXgTxjAObzSdA1OWC7BPGooyTVksTfqjeHg4QtxImqyJ89BRoUCx0YDDZk/o7TEHgcyz0nBdY99JPDzmfa+Jy/uasDsnawiXiQ2zFT5SxHJvyH+YeFE7H0gShEAHCfoqHj4ahXQh3oeHHPDoGc27paej6LfE1x21OPJDxmREUlDIpjATjT6haRBOCjxyUO///Ue9sKqZAgpng/7P5StiMzMSGD28jCOQ5/x6ncsrq2nej5unQ66Sw8PRGgLfB2R7H6pk1L2bjfNaN6QIjKIk4sMTCuwMOYXvdGDUOYZcQWFEOG19K46vkH/QLwwtz2MOMxyeyff4MHwKsQuiDEQWxWtakSmSTKdZ0BlajgTxrM/C8S+tKemIVa9Y/RqOs4/i4UlQ1trJhKQZpuDTmk4bd5Ly6L6kBNAgB2lgwzecWqdNhuqyMYaGOrFqGM88NNACvKwnzKhMsbrB8ZaZNPcq0ZvGbXb1/8nbo9+QtyZp7z7DXxY63EXQdRFKSQfSd5ux9nHsQI3hHNW7aFvsmwfx4JAC/PJ2bB3zzIoKd5tFpnRDcMsAClBDtlh8QUEiJkeEEFjCGo5IA97m7dqtmyqQYqfo8Abhtykq7TxY5xQ4STm9E1EaEAaQNJbZAqMOyJZKlGuSy2KSga+dgqC6etKx0c8/PcSuaPvqtF8aC0qqP6hO9Pq7Zxi03X6K0T01eYZBuodTj7HEuSgLWbitRIt2juPBgEgVPWYg7jzdn957Nfox+xR1+7PcTDrRH5v7UNqJUXt9yuLFw0QMLYvi4VDIqrO5Ko2rz16OdgXqNwp0KmLbGI0jXhMejKBILVNVefGcPwBKp8ivC56N+WL2o+3TyI52+FjU3djEqz+dNfMJoCo2+8UfxMfNYFzvHm120yT8tMeXgcFa6OQk0UBt7B/taiPnSCw7M0qSiPdHn/a2jNgWSWZ8SWgwokrO96OSbi8kcIZoq2svqas+NacKkY4agsyJyOqich0b0wnWqrF0qdYN+XgY/VxGt5WgbGSNY37gz0zaaY7zqxijD4gXXm1d2/ef0XNlyfZAmnfj1+3u8mWXODqXZp3fEv27u8dOAYd0OQ6ZrOhyhtqIeGvDvQKx2V3JQW9XIIBjZGKMkZWNp+1bUAEQROxr/Eve2lrF1HCfobDVFwlsoWHSMGqwc3/DJt+8Ov+hIagxFwSREgx8ReOm5YuER3RbwsTHAQG+Y/Yr1qy310LIWcNKsaPS2MjcNr8m0SLwVBF4KoQ4vWxZE+GRppM0/enFMKm6MqKxtuXP67jHSIjM6BqmIu5DHcujyJQyp/miMsI2P7cq3JGFmz/GNhForGOE9bZOKQasNbDwDXsn5Ddh3rVMzZ1xhNZ8WrQ0OtpwRb49gEGNOhYNw4+OhUEEc1mJK+3nqCnKO7uKzTLP6eS90Ag6fDcTHp4ZrwHPsWTCDFYwIeKHIVcM3+EB0doOGvPJu29fvO29Pn/34/Mn7/7We/v8zds3zZ9utptzqym50+5pzBjnRDpTHoYO7Z6hdsvftBPt+z9R2lzrexsAAA==', 'VERIFICATION-CPU-LOCALE.json': 'H4sIAAAAAAAC/1WPO24DMQxE+z3FwrUbd0EuQ3ClgUJYP5NUNnGQu0dGAGPdEm/ecH6WdT1x8MGZzDnhciFXlgql0AcpbBTQJr6LgXCb4Ol9dR04P6I7JH24UesuRe4zpTUR10ihjepQO9Kp5UhvSbkYdW2O4IhHQNHB8/YPTUfpGS6tGhW26ys8Mk0qCqpTbvvsoorELp942DfeJIt/HyOdY5T5IJqRdQSZs71dMf34CnnE14YwIj+31+YUMkt5Msvv8gch85cIQAEAAA=='}\nHASHES11={'mandia_stage11.py': '1666e8bd5ba30313fd5c4f9fad538f08e0c3b784f1b31a7479f6252e50fd7049', 'nouveaux-tests.jsonl': '731bae931a6141b1392fd1862e4ddafe0d6bf73d6c4d37ee6f7702ea6ce767cd', 'AUDIT-NOUVEAUX-TESTS.json': 'f1c863f8c9c4c49ee93e3e99018899814d1073dbf69473a8dbc3ed4497a942e9', 'METHODE-ET-SOURCES.txt': '142e2327375b156b208ef253021ecd0e43cc28a3fefe6a0438086ee12bbf9df6', 'preparer-tests.py': 'e2e363c22a99612391f7624bdcd87b19c31c8fc9f761705d099a5fae190648fa', 'VERIFICATION-CPU-LOCALE.json': 'ec9f65924e117c91d517b0267843bd11f3d24c290bceeafaa1476112311014f3'}\nfor name,encoded in ASSETS11.items():\n data=gzip.decompress(base64.b64decode(encoded));assert hashlib.sha256(data).hexdigest()==HASHES11[name]\n dest=ROOT11/name\n if dest.exists():assert hashlib.sha256(dest.read_bytes()).hexdigest()==HASHES11[name], 'Preserve prior run: source changed '+name\n else:dest.write_bytes(data)\ntry:import tokenizers,sacrebleu\nexcept ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1','sacrebleu==2.5.1'])\nfor rel in ['mandia-compact-s2s-etape6/code','mandia-compact-s2s-etape7','mandia-compact-s2s-etape10','mandia-compact-s2s-etape11']:sys.path.insert(0,str(ROOT11.parent/rel))\nimport mandia_stage11\nimportlib.reload(mandia_stage11)\nprint('MANDIA_ETAPE11_DEBUT : 3 essais contrôlés,300 étapes chacun, source française et répétitions, Drive obligatoire.',flush=True)\nreport11=mandia_stage11.main()\n",globals())
finally:
 if 'ROOT11' in globals() and ROOT11.exists():
  (ROOT11/'journal-entrainement-reel.txt').write_text(capture11.stdout+capture11.stderr)
 capture11.show()


In [ ]:
from IPython.display import display,Markdown
import pathlib,json
root=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape11')
r=json.loads((root/'rapport-etape11.json').read_text());names=['parent']+list(r['arms'])
results=[r['before']]+[r['arms'][k]['test'] for k in r['arms']]
lines=['## Étape11 terminée : trois essais à20,86 M de paramètres',f"Sélection sur validation : **{r['selected_by_validation']}**. Les trois entraînements de300 étapes sont terminés ; meilleurs poids et reprise300 conservés séparément dans Drive.",'','| Mesure | Parent | CE | Contraste | Contraste+UL |','|---|---:|---:|---:|---:|']
for k,label in [('fresh_translation','chrF++ nouvelles traductions30'),('fresh_written_reply','chrF++ nouvelles répliques6'),('fresh_authored_advice','chrF++ questions pratiques3')]:
 lines.append('| '+label+' | '+' | '.join(f"{x[k]['generation']['chrF_pp']:.2f}" for x in results)+' |')
lines.append('| Répétition mots4, nouvelles traductions | '+' | '.join(f"{x['fresh_translation']['generation']['mean_repeated_word4']:.1%}" for x in results)+' |')
lines.append('| Écart NLL bonne/mauvaise source, traductions | '+' | '.join(f"{x['source_fresh_translation']['gap']:.3f}" for x in results)+' |')
lines+=['','**Lecture prudente :** les répétitions sont mesurées avec le même décodage glouton, sans blocage. Une baisse des boucles ou une hausse de chrF++ ne prouve pas la compréhension. Examiner aussi pertes, sorties vides, EOS et longueur dans le rapport.','', '**Données :** quatre nouveaux documents réservés aux tests, français traduit/adapté par assistant, cibles baoulé publiées dans SMOL. Six répliques de deux dialogues écrits ; trois questions pratiques construites. Pas de nouvelles conversations enregistrées ni de validation native. Les anciens40 cas mesurent une régression sur familles connues.','', '**Drive :** dossiers ce_control, source_contrastive, source_contrastive_ul ; checkpoint-last.pt contient AdamW et les RNG. Aucun poids lourd téléchargé sur le Mac. La voix reste une étape ultérieure.']
text='\n'.join(lines);(root/'RESULTATS.md').write_text(text+'\n');display(Markdown(text))
print('ETAPE11_RESULTATS_AFFICHES',json.dumps({'selected':r['selected_by_validation'],'steps':{k:v['training']['best_step'] for k,v in r['arms'].items()}},ensure_ascii=False))


In [ ]:
from IPython.utils.capture import capture_output
import pathlib
ROOT11=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape11')
(ROOT11/'diagnostic-source-complet.py').write_text("# Supplement only: more cross-document pairs, no retraining or reselection.\nimport pathlib,sys,json,gc\nROOT=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape11')\nfor rel in ['mandia-compact-s2s-etape6/code','mandia-compact-s2s-etape7','mandia-compact-s2s-etape10','mandia-compact-s2s-etape11']:sys.path.insert(0,str(ROOT.parent/rel))\nimport torch\nimport mandia_stage4 as s4\nimport mandia_stage6 as s6\nimport mandia_stage11 as s11\nfrom tokenizers import Tokenizer\nfrom mandia_seq2seq import MandiaS2S,S2SConfig\ns4.DEVICE=torch.device('cuda');tok=Tokenizer.from_file(str(s4.ROOT/'tokenizer.json'))\nitems=s11.encode([json.loads(x) for x in (ROOT/'nouveaux-tests.jsonl').read_text().splitlines()],tok)\ngroups={k:[x for x in items if x['row']['domain']==k] for k in ['fresh_translation','fresh_written_reply']}\n@torch.inference_mode()\ndef diagnose(m,x):\n m.eval();n=len(x)\n def valid(shift):return [a['row']['source_document']!=x[(i-shift)%n]['row']['source_document'] and s4.canonical(a['row']['answer'])!=s4.canonical(x[(i-shift)%n]['row']['answer']) for i,a in enumerate(x)]\n # Choose on document identities/reference inequality, never model scores.\n shift=max(range(1,n),key=lambda s:sum(valid(s)));mask=torch.tensor(valid(shift),device=s4.DEVICE)\n a,b,y,_=s4.batch(x,s4.DEVICE)\n with s4.context():p=s6.row_nll(m(a,b),y);z=s6.row_nll(m(a.roll(shift,0),b),y)\n return dict(rows=n,valid_pairs=int(mask.sum()),rotation=shift,matched_nll=float(p[mask].mean()),permuted_nll=float(z[mask].mean()),gap=float((z-p)[mask].mean()),pairs_favoring_correct_source=int((z[mask]>p[mask]).sum()))\npaths={'parent':ROOT.parent/'mandia-compact-s2s-etape10/translation_dialogue/model.pt'}\npaths.update({k:ROOT/k/'model.pt' for k in ['ce_control','source_contrastive','source_contrastive_ul']});out={}\nfor name,path in paths.items():\n p=torch.load(path,map_location='cpu',weights_only=True);m=MandiaS2S(S2SConfig(**p['config'])).cuda();m.load_state_dict(p['state_dict']);del p\n out[name]={k:diagnose(m,x) for k,x in groups.items()};del m;gc.collect();torch.cuda.empty_cache()\nresult=dict(method='metadata-only maximum-valid deterministic cyclic rotation; per-row NLL, padding ignored; same pairs for all four models',original_report='Original one-position permutation yields only2 cross-document pairs per test group, retained as historical evidence; use this supplement for broader source diagnostic',selection_unchanged=True,models=out,limits=['only2 source document families per group','diagnostic only, not semantic/native correctness','28 of30 valid translation pairs because source families have unequal16/14 sizes','no new training or checkpoint selection'])\n(ROOT/'DIAGNOSTIC-SOURCE-COMPLET.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\\n')\nprint('SOURCE_DIAGNOSTIC_COMPLETE',json.dumps(result,ensure_ascii=False),flush=True)\n")
with capture_output() as diagCapture11:
 exec((ROOT11/'diagnostic-source-complet.py').read_text(),globals())
(ROOT11/'journal-diagnostic-reel.txt').write_text(diagCapture11.stdout+diagCapture11.stderr)
diagCapture11.show()


In [ ]:
#@title Comparer les checkpoints conservés — français → baoulé (expérimental)
texte_fr = "Les enfants vont à l'école, tandis que les adultes vont au travail." #@param {type:"string"}
mode = "Traduire" #@param ["Traduire", "Répondre"]
checkpoint = "Parent étape10" #@param ["Parent étape10", "CE témoin", "Contraste — sélection validation", "Contraste + UL — expérimental"]
contexte_fr = "" #@param {type:"string"}
from google.colab import drive
import pathlib,sys,json,hashlib,subprocess
if not pathlib.Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
runtime=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime');root=runtime/'mandia-compact-s2s-etape11'
try:import torch,tokenizers
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1'])
import torch
from tokenizers import Tokenizer
sys.path.insert(0,str(runtime/'mandia-compact-s2s-etape6/code'));sys.path.insert(0,str(root))
from mandia_seq2seq import MandiaS2S,S2SConfig
from traduire_texte import predict
if checkpoint=='Parent étape10':weightroot=runtime/'mandia-compact-s2s-etape10';relative='translation_dialogue/model.pt'
else:
 weightroot=root;relative={'CE témoin':'ce_control','Contraste — sélection validation':'source_contrastive','Contraste + UL — expérimental':'source_contrastive_ul'}[checkpoint]+'/model.pt'
path=weightroot/relative;receipt=json.loads((weightroot/'SAUVEGARDE-VERIFIEE.json').read_text());expected=next(x['sha256'] for x in receipt['files'] if x['path']==relative)
assert hashlib.sha256(path.read_bytes()).hexdigest()==expected
p=torch.load(path,map_location='cpu',weights_only=True);tp=runtime/'mandia-compact-s2s-etape4/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest()==p['tokenizer_sha256']
tok=Tokenizer.from_file(str(tp));device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=MandiaS2S(S2SConfig(**p['config'])).to(device);model.load_state_dict(p['state_dict']);model.eval();del p
result=predict(model,tok,texte_fr,mode='translate' if mode=='Traduire' else 'reply',context=contexte_fr)
print('Checkpoint :',checkpoint,'; paramètres :',sum(v.numel() for v in model.parameters()))
print('Baoulé expérimental :',result['text_bci'])
if not result['all_chunks_ended_eos']:print('Sortie inachevée : la limite de génération a été atteinte.')
print('Aucun candidat étape11 promu : les dialogues régressent. Qualité linguistique à vérifier. Segments traités :',len(result['chunks']))
del model
